# Implementing multi head attention with tensors
- We can implement the idea of weight split in MHA to avoid High Time Complexity.

# Steps:-
---

**Step 1:** Reduce the projection dim to match desired output dim

**Step 2:** Use a Linear layer to combine head outputs

**Step 3:** Tensor shape: `(b, num_tokens, d_out)`

**Step 4:** We implicitly split the matrix by adding a`num_heads` dimension. Then we unroll last dim:
`(b, num_tokens, d_out) → (b, num_tokens, num_heads, head_dim)`

**Step 5:** Transpose from shape
`(b, num_tokens, num_heads, head_dim)`to`(b, num_heads ,num_tokens, head_dim)`


**Step 6:** Compute dot product for each head

**Step 7:** Mask truncated to the number of tokens

**Step 8:** Use the mask to fill attention scores

**Step 9:** Tensor shape: `(b, num_tokens, n_heads, head_dim)`

**Step 10:** Combine heads, where `self.d_out = self.num_heads * self.head_dim`

**Step 11:** Add an optional linear projection

## Step 1: Start with the input

We define the input tensor `x` with shape:

- **b (batch size)** = 1  
- **num_tokens** = 3  
- **d_in (input dimension)** = 6  

So the tensor shape is: **(1, 3, 6)**  

In [8]:
import torch

# Input tensor
x = torch.tensor(
    [
        [
            [1.0, 2.0, 3.0, 4.0, 5.0, 6.0],  # the
            [6.0, 5.0, 4.0, 3.0, 2.0, 1.0],  # Kid
            [1.0, 1.0, 1.0, 1.0, 1.0, 1.0],  # smiles
        ]
    ]
)

# Check shape
print(x.shape)
batch_size, num_tokens, d_in = x.shape
print(batch_size, num_tokens, d_in)

torch.Size([1, 3, 6])
1 3 6


## Step 2: Decide `d_out` and `num_heads`

We choose:
- **d_out** (output dimension) = 6  
- **num_heads** = 2  

Each head will therefore have:
- **head_dim = d_out / num_heads = 3**


In [9]:
# Output dimension and number of attention heads
d_out = 6
num_heads = 2

# Dimension per head
head_dim = d_out // num_heads

print(d_out, num_heads, head_dim)

6 2 3


## Step 3: Initialize `Wq`, `Wk`, `Wv`

We initialize the Query, Key, and Value projection matrices.

- Input dimension = 6  
- Output dimension = 6  

So each matrix has shape **(6 × 6)**.


In [10]:
# Initialize weight matrices
torch.manual_seed(0)  # for Reporducibility
Wq = torch.randn(d_in, d_in)
Wk = torch.randn(d_in, d_in)
Wv = torch.randn(d_in, d_in)

# Check shapes
print(Wq.shape, Wk.shape, Wv.shape)

torch.Size([6, 6]) torch.Size([6, 6]) torch.Size([6, 6])


## Step 4: Calculate Q, K, V

We compute the **Query (Q)**, **Key (K)**, and **Value (V)** matrices by
multiplying the input tensor `x` with the corresponding projection matrices.

Formally:

- **Q = X · Wq**
- **K = X · Wk**
- **V = X · Wv**

Where:
- `X` has shape **(1, 3, 6)**
- `Wq`, `Wk`, `Wv` have shape **(6, 6)**

The resulting tensors **Q, K, V** each have shape **(1, 3, 6)**.

In [ ]:
# Compute Query, Key, and Value matrices
Q = x @ Wq
K = x @ Wk
V = x @ Wv
print(Q.shape, K.shape, V.shape)

print("Q Matrix:")
print(Q)
print("K Matrix:")
print(K)
print("V Matrix:")
print(V)

torch.Size([1, 3, 6]) torch.Size([1, 3, 6]) torch.Size([1, 3, 6])
Q Matrix: 
tensor([[[ -9.0244, -11.7287,  15.5360,  -1.4474,  -4.5326,   9.4674],
         [ -8.0564, -13.2309,   8.2228,  -8.9680,   3.1995,   4.8321],
         [ -2.4401,  -3.5657,   3.3941,  -1.4879,  -0.1904,   2.0428]]])
K Matrix: 
tensor([[[  8.2602,  14.1116,  -5.0345, -16.4865,  -2.9948,   8.3139],
         [ -6.1188,  -0.1587,  -5.0885, -14.3014,   4.9540,   5.6093],
         [  0.3059,   1.9933,  -1.4461,  -4.3983,   0.2799,   1.9890]]])
V Matrix: 
tensor([[[ 0.5076, -3.4353,  1.8576,  2.8041,  8.9427, 13.1841],
         [-1.9113, -3.6934,  1.8502,  1.7622,  1.6981,  3.0978],
         [-0.2005, -1.0184,  0.5297,  0.6523,  1.5201,  2.3260]]])


# Step 5: Unroll the last Dimensions of `Q`,`K`,`V` to include `num_heads`

We compute the **Query (Q)**, **Key (K)**, and **Value (V)** matrices by
multiplying the input tensor `x` with the corresponding projection matrices.

Formally:

- **Q = X · Wq**
- **K = X · Wk**
- **V = X · Wv**

### Q, K, V Transformation

**Shape Change:** $\text{3D} \rightarrow \text{4D}$

$$1 \times 3 \times 6 \quad \rightarrow \quad 1 \times 3 \times 2 \times 3$$

**Dimension Mapping:**
`[batch, num_tokens, num_heads, head_dim]`

> **Note:** The embedding dimension ($6$) is split into `num_heads` ($2$) $\times$ `head_dim` ($3$).

In [12]:
num_heads = 2
head_dim = 3

Q = Q.view(1, 3, num_heads, head_dim)
K = K.view(1, 3, num_heads, head_dim)
V = V.view(1, 3, num_heads, head_dim)

print("Q After Unrolling:")
print(Q)
print("K After Unrolling:")
print(K)
print("V After Unrolling:")
print(V)

Q After Unrolling:
tensor([[[[ -9.0244, -11.7287,  15.5360],
          [ -1.4474,  -4.5326,   9.4674]],

         [[ -8.0564, -13.2309,   8.2228],
          [ -8.9680,   3.1995,   4.8321]],

         [[ -2.4401,  -3.5657,   3.3941],
          [ -1.4879,  -0.1904,   2.0428]]]])
K After Unrolling:
tensor([[[[  8.2602,  14.1116,  -5.0345],
          [-16.4865,  -2.9948,   8.3139]],

         [[ -6.1188,  -0.1587,  -5.0885],
          [-14.3014,   4.9540,   5.6093]],

         [[  0.3059,   1.9933,  -1.4461],
          [ -4.3983,   0.2799,   1.9890]]]])
V After Unrolling:
tensor([[[[ 0.5076, -3.4353,  1.8576],
          [ 2.8041,  8.9427, 13.1841]],

         [[-1.9113, -3.6934,  1.8502],
          [ 1.7622,  1.6981,  3.0978]],

         [[-0.2005, -1.0184,  0.5297],
          [ 0.6523,  1.5201,  2.3260]]]])


### Step 6: Group matrices by number of heads

**1. Initial Reshape (Split Heads):**
$$Q, K, V: \rightarrow 1 \times 3 \times 2 \times 3$$
`[batch, num_tokens, num_heads, head_dim]`

**2. Transpose (Group by Heads):**
*We swap the `num_tokens` and `num_heads` dimensions so that each head can work on all tokens in parallel.*

$$Q, K, V: \rightarrow 1 \times 2 \times 3 \times 3$$
`[batch, num_heads, num_tokens, head_dim]`

In [14]:
# Swap dimension 1 (tokens) with dimension 2 (heads)
Q = Q.transpose(1, 2)
K = K.transpose(1, 2)
V = V.transpose(1, 2)
print("Q after Grouping of heads:")
print(Q)
print("K after Grouping of Heads:")
print(K)
print("V after Grouping of Heads:")
print(V)
print(Q.shape)

Q after Grouping of heads:
tensor([[[[ -9.0244, -11.7287,  15.5360],
          [ -1.4474,  -4.5326,   9.4674]],

         [[ -8.0564, -13.2309,   8.2228],
          [ -8.9680,   3.1995,   4.8321]],

         [[ -2.4401,  -3.5657,   3.3941],
          [ -1.4879,  -0.1904,   2.0428]]]])
K after Grouping of Heads:
tensor([[[[  8.2602,  14.1116,  -5.0345],
          [-16.4865,  -2.9948,   8.3139]],

         [[ -6.1188,  -0.1587,  -5.0885],
          [-14.3014,   4.9540,   5.6093]],

         [[  0.3059,   1.9933,  -1.4461],
          [ -4.3983,   0.2799,   1.9890]]]])
V after Grouping of Heads:
tensor([[[[ 0.5076, -3.4353,  1.8576],
          [ 2.8041,  8.9427, 13.1841]],

         [[-1.9113, -3.6934,  1.8502],
          [ 1.7622,  1.6981,  3.0978]],

         [[-0.2005, -1.0184,  0.5297],
          [ 0.6523,  1.5201,  2.3260]]]])
torch.Size([1, 3, 2, 3])


### Step 7: Find attention scores

To calculate the attention scores (dot product), we need to align the dimensions of **Q** and **K**. Specifically, we transpose the last two dimensions of **K**.

**1. Query (Q) Shape:**
$$Q: \rightarrow 1 \times 2 \times 3 \times 3$$
`[batch, num_heads, num_tokens, head_dim]`

**2. Key (K) Transposed Shape:**
$$K^T: \rightarrow 1 \times 2 \times 3 \times 3$$
`[batch, num_heads, head_dim, num_tokens]`

In [15]:
# We transpose the last two dimensions of K to allow for matrix multiplication
# K was [1, 2, 3, 3] -> becomes [1, 2, 3, 3] (conceptually swapping 3 and 3)
K_T = K.transpose(2, 3)

print("Key Transposed :")
print(K_T)
print("Key Transposed shape", K_T.shape)

Key Transposed :
tensor([[[[  8.2602, -16.4865],
          [ 14.1116,  -2.9948],
          [ -5.0345,   8.3139]],

         [[ -6.1188, -14.3014],
          [ -0.1587,   4.9540],
          [ -5.0885,   5.6093]],

         [[  0.3059,  -4.3983],
          [  1.9933,   0.2799],
          [ -1.4461,   1.9890]]]])
Key Transposed shape torch.Size([1, 3, 3, 2])


### Step 8: Calculate Attention Scores

We compute the dot product of $Q$ and $K^T$.

**Formula:**
$$\text attn\_scores = Q \cdot K^T$$

**Resulting Dimensions:**
The result represents the relationship between every token and every other token, for each head.
$$\rightarrow \quad [batch\_size, num\_heads, num\_tokens, num\_tokens]$$

In [17]:
# Perform matrix multiplication
attn_scores = Q @ K_T

print("Attention scores shape:", attn_scores.shape)
print("Attention scores:\n", attn_scores)

Attention scores shape: torch.Size([1, 3, 2, 2])
Attention scores:
 tensor([[[[-318.2692,  313.0702],
          [-123.5821,  116.1476]],

         [[   9.5538,   95.7964],
          [  29.7783,  171.2106]],

         [[ -12.7621,   16.4853],
          [  -3.7889,   10.5541]]]])


###  Step 9: Apply Causal Masking

In a decoder-only model (like GPT), we must mask future tokens so the model cannot "cheat" by seeing what comes next.

1.  **Create Mask:** We generate an upper triangular matrix of ones.
2.  **Apply Mask:** We replace the positions where the mask is `1` (or `True`) with negative infinity (`-inf`). This ensures that after softmax, these positions will have a probability of `0`.

In [18]:
seq_len = attn_scores.shape[-1]

# Create upper triangular mask
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
print("Causal mask:\n", mask)

# Apply mask (fill with -inf)
attn_scores = attn_scores.masked_fill_(mask, -torch.inf)

print("\nAttention scores after masking:\n", attn_scores)

Causal mask:
 tensor([[False,  True],
        [False, False]])

Attention scores after masking:
 tensor([[[[-318.2692,      -inf],
          [-123.5821,  116.1476]],

         [[   9.5538,      -inf],
          [  29.7783,  171.2106]],

         [[ -12.7621,      -inf],
          [  -3.7889,   10.5541]]]])


### Scale and Softmax

Finally, we convert the raw scores into probabilities (attention weights).

1.  **Scale:** Divide the scores by $\sqrt{d_{head}}$ (where $d_{head}$ is the head dimension, e.g., 3). This prevents the gradients from vanishing when the dimensions are large.
2.  **Softmax:** Apply softmax to the last dimension to ensure the weights sum to 1.

In [19]:
torch.set_printoptions(precision=3, sci_mode=False)  # Turn off scientific notation

head_dim = 3  # Get head_dim from K tensor size (or define explicitly)

# Scale by square root of head dimension and apply Softmax
attn_weights = torch.softmax(attn_scores / head_dim**0.5, dim=-1)

print("Attention weights shape:", attn_weights.shape)
print("Attention weights:\n", attn_weights)

Attention weights shape: torch.Size([1, 3, 2, 2])
Attention weights:
 tensor([[[[1.000, 0.000],
          [0.000, 1.000]],

         [[1.000, 0.000],
          [0.000, 1.000]],

         [[1.000, 0.000],
          [0.000, 1.000]]]])


### Apply Dropout

We apply dropout to the attention weights. This randomly zeroes out some of the probabilities during training to prevent overfitting.

* **Rate:** 0.1 (10% of the elements will be zeroed out).

In [20]:
dropout = torch.nn.Dropout(0.1)  # You can adjust the dropout rate
attn_weights = dropout(attn_weights)
print("Attention weights after dropout:\n", attn_weights)

Attention weights after dropout:
 tensor([[[[1.111, 0.000],
          [0.000, 1.111]],

         [[1.111, 0.000],
          [0.000, 1.111]],

         [[1.111, 0.000],
          [0.000, 1.111]]]])


### Step 10: Calculate context vectors

We multiply the attention weights by the Value matrix ($V$) to get the final context vectors.

**1. Attention weights:**
`[batch_size, num_heads, num_tokens, num_tokens]`

**2. Value matrix:**
$$1 \times 2 \times 3 \times 3$$
`[batch, num_heads, num_tokens, head_dim]`

In [21]:
# Calculate context vectors
# Shape: [batch, heads, tokens, tokens] @ [batch, heads, tokens, head_dim]
# Result: [batch, heads, tokens, head_dim]
context_vectors = attn_weights @ V

print("Context vectors shape:", context_vectors.shape)
print("Context vectors:\n", context_vectors)

Context vectors shape: torch.Size([1, 3, 2, 3])
Context vectors:
 tensor([[[[ 0.564, -3.817,  2.064],
          [ 3.116,  9.936, 14.649]],

         [[-2.124, -4.104,  2.056],
          [ 1.958,  1.887,  3.442]],

         [[-0.223, -1.132,  0.589],
          [ 0.725,  1.688,  2.584]]]])


# Step 11: Reformat and Concatenate Heads

This block of code implements the final stage of the **Multi-Head Attention** mechanism. We are taking the independent attention results from each head and merging them back into a single vector representation for the next layer.

### 1. The Transpose
First, we need to reorder the dimensions. We swap the `num_heads` and `num_tokens` dimensions to align the data correctly for flattening.

**2. context vector matrix after swapping:**
$$1 \times 6 \times 3$$

In [22]:
context_vectors = context_vectors.transpose(1, 2)
print(context_vectors)
print(context_vectors.shape)

tensor([[[[ 0.564, -3.817,  2.064],
          [-2.124, -4.104,  2.056],
          [-0.223, -1.132,  0.589]],

         [[ 3.116,  9.936, 14.649],
          [ 1.958,  1.887,  3.442],
          [ 0.725,  1.688,  2.584]]]])
torch.Size([1, 2, 3, 3])


In [23]:
context_vectors = context_vectors.reshape(batch_size, num_tokens, num_heads * head_dim)
print("context vector after concatinationg head\n", context_vectors)
print("shape of context Vector after Concatinating\n", context_vectors.shape)

# dimensions=(batch,num_tokens,d_out)

context vector after concatinationg head
 tensor([[[ 0.564, -3.817,  2.064, -2.124, -4.104,  2.056],
         [-0.223, -1.132,  0.589,  3.116,  9.936, 14.649],
         [ 1.958,  1.887,  3.442,  0.725,  1.688,  2.584]]])
shape of context Vector after Concatinating
 torch.Size([1, 3, 6])


# We Can write all the steps inside a Single class

In [25]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = (
            d_out // num_heads
        )  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine heads
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)  # optional projection

        return context_vec

In [26]:
torch.manual_seed(123)

# Define the tensor with 3 rows and 6 columns
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  # Row 1
        [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  # Row 2
        [0.77, 0.25, 0.10, 0.05, 0.80, 0.55],
    ]  # Row 3
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

batch_size, context_length, d_in = batch.shape
d_out = 6
context_length = inputs.shape[0]
dropout = 0.1
num_heads = 2
mha = MultiHeadAttention(d_in, d_out, context_length, dropout, num_heads)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 3, 6])
tensor([[[ 0.167, -0.077,  0.051,  0.020, -0.327, -0.246],
         [ 0.117, -0.041,  0.073, -0.027, -0.328, -0.298],
         [ 0.072, -0.081, -0.010, -0.024, -0.336, -0.314]],

        [[ 0.074, -0.204, -0.127,  0.066, -0.405, -0.348],
         [ 0.117, -0.041,  0.073, -0.027, -0.328, -0.298],
         [ 0.094, -0.078,  0.002, -0.058, -0.303, -0.287]]],
       grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 3, 6])
